# Inventario de separadores

Este caderno executa uma auditoria read-only dos Parquets completos de `textos_parlamentares/v1` no Google Drive. Ele gera relatorios para revisar separadores antes de qualquer limpeza automatica do campo `texto`.


## 1. Montar Google Drive


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Ambiente fora do Colab; mantendo caminhos locais se existirem.')


Ambiente fora do Colab; mantendo caminhos locais se existirem.


## 2. Preparar repositorio


In [2]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/falando_nela')
REPO_URL = 'https://github.com/pedroblancode/falando_nela.git'

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = Path.cwd()
    print('Usando repositorio local:', REPO_DIR)

print('Repositorio:', Path.cwd())


Usando repositorio local: /Users/pedblan/PycharmProjects/falando_nela/notebooks/processamento
Repositorio: /Users/pedblan/PycharmProjects/falando_nela/notebooks/processamento


## 3. Instalar dependencias


In [3]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=15,<23', 'pandas>=2,<3'],
    check=True,
)



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/Users/pedblan/PycharmProjects/falando_nela/.venv/bin/python', '-m', 'pip', 'install', '-q', 'pyarrow>=15,<23', 'pandas>=2,<3'], returncode=0)

## 4. Definir caminhos


In [4]:
import os
from datetime import datetime
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)
PARQUET_ROOT = DATA_ROOT / 'processed' / 'textos_parlamentares' / 'v1' / 'parquet'
RUN_ID = 'separadores-textos-v1-' + datetime.now().strftime('%Y%m%d')
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'audits' / 'separadores' / RUN_ID

print('Parquets:', PARQUET_ROOT)
print('Run ID:', RUN_ID)
print('Saida:', OUTPUT_ROOT)


Parquets: /content/drive/MyDrive/falando_nela/data/processed/textos_parlamentares/v1/parquet
Run ID: separadores-textos-v1-20260527
Saida: /content/drive/MyDrive/falando_nela/data/processed/audits/separadores/separadores-textos-v1-20260527


## 5. Conferir Parquets disponiveis


In [5]:
parquets = sorted(PARQUET_ROOT.glob('*.parquet'))
print('Parquets encontrados:', len(parquets))
for path in parquets:
    print('-', path.name)
if not parquets:
    raise FileNotFoundError(f'Nenhum Parquet encontrado em {PARQUET_ROOT}')


Parquets encontrados: 0


FileNotFoundError: Nenhum Parquet encontrado em /content/drive/MyDrive/falando_nela/data/processed/textos_parlamentares/v1/parquet

## 6. Executar inventario


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    '-m',
    'processamento.inventario_separadores',
    '--profile',
    'colab',
    '--run-id',
    RUN_ID,
    '--overwrite',
]
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Inventario falhou com codigo {result.returncode}')


## 7. Abrir manifest


In [ ]:
import json
from pprint import pprint

manifest_path = OUTPUT_ROOT / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pprint(manifest)


## 8. Resumo de separadores


In [ ]:
import pandas as pd

separadores_path = OUTPUT_ROOT / 'separadores_resumo.csv'
separadores = pd.read_csv(separadores_path)
print('Linhas:', len(separadores))
display(separadores.head(50))


## 9. Resumo de parenteses taquigraficos


In [ ]:
parenteticos_path = OUTPUT_ROOT / 'parenteticos_resumo.csv'
parenteticos = pd.read_csv(parenteticos_path)
print('Linhas:', len(parenteticos))
display(parenteticos.head(50))


## 10. Exemplos para auditoria


In [ ]:
exemplos_path = OUTPUT_ROOT / 'separadores_exemplos.jsonl'
exemplos = pd.read_json(exemplos_path, lines=True)
print('Exemplos:', len(exemplos))
cols = [
    'source',
    'dataset',
    'ano',
    'action',
    'kind',
    'separator_normalized',
    'separator_text',
    'texto_id',
    'trailing_chars',
]
display(exemplos[[col for col in cols if col in exemplos.columns]].head(50))


## 11. Inspecionar um exemplo


In [ ]:
EXAMPLE_INDEX = 0

if exemplos.empty:
    print('Nenhum exemplo encontrado.')
else:
    row = exemplos.iloc[EXAMPLE_INDEX]
    print('texto_id:', row.get('texto_id'))
    print('acao:', row.get('action'))
    print('tipo:', row.get('kind'))
    print('separador:', row.get('separator_text'))
    print('\nAntes:\n')
    print(row.get('context_before', ''))
    print('\nDepois:\n')
    print(row.get('context_after', ''))


## 12. Amostra para revisão por IA

O inventário também gera uma amostra determinística de 0,1% por `source/dataset/ano`, com mínimo de 1 texto por estrato, para revisão posterior por IA com resposta estruturada. Esta célula apenas lê os arquivos gerados.

In [ ]:

ai_sample_path = OUTPUT_ROOT / 'amostra_ia_textos.jsonl'
ai_prompt_path = OUTPUT_ROOT / 'amostra_ia_prompt.md'
ai_schema_path = OUTPUT_ROOT / 'amostra_ia_schema.json'

ai_sample = pd.read_json(ai_sample_path, lines=True)
print('Textos na amostra de IA:', len(ai_sample))
display(ai_sample[['custom_id', 'source', 'dataset', 'ano', 'texto_id', 'texto_tamanho_original', 'texto_truncado']].head(50))


## 13. Prompt e schema estruturado

In [ ]:

import json

print('Prompt para revisão por IA:\n')
print(ai_prompt_path.read_text(encoding='utf-8'))
print('\nSchema JSON:\n')
schema = json.loads(ai_schema_path.read_text(encoding='utf-8'))
print(json.dumps(schema, ensure_ascii=False, indent=2))


## 14. Conferencia final

A saida desta auditoria fica em `processed/audits/separadores/{RUN_ID}`. Revise os separadores `hard_cut` e `review`, e use a amostra de IA apenas como apoio para decidir regras futuras. Este caderno nao limpa nem reescreve o campo `texto`.